In [33]:
import pandas as pd
import os
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [13]:
## Bonne implementaion de l'upload

import kagglehub
import shutil
import os

# 1. Téléchargement via kagglehub (va dans le cache par défaut)
cache_path = kagglehub.dataset_download("puneet6060/intel-image-classification")
print("Chemin temporaire :", cache_path)

# 2. Le dossier où vous voulez vraiment vos données
dossier_final = "./my_intel_dataset"

# 3. On copie les données vers le dossier final
if os.path.exists(dossier_final):
    # Si le dossier existe déjà, on le supprime pour éviter les conflits
    shutil.rmtree(dossier_final)

# Utiliser shutil.copytree au lieu de shutil.move car cache_path est en lecture seule
shutil.copytree(cache_path, dossier_final)

print("Succès ! Le dataset se trouve maintenant directement dans :", dossier_final)

Using Colab cache for faster access to the 'intel-image-classification' dataset.
Chemin temporaire : /kaggle/input/intel-image-classification
Succès ! Le dataset se trouve maintenant directement dans : ./my_intel_dataset


Create the label file

In [14]:
#dataset_train_path = '/Users/laurageneaulibourne/Downloads/S4/ia/exam/dataset/seg_train/seg_train'
dataset_train_path="/content/my_intel_dataset/seg_train/seg_train"

#dataset_test_path = '/Users/laurageneaulibourne/Downloads/S4/ia/exam/dataset/seg_test/seg_test'
dataset_test_path='/content/my_intel_dataset/seg_test/seg_test'

classe = {'buildings' : 0, 'forest' : 1, 'glacier' : 2, 'mountain' : 3, 'sea' : 4, 'street' : 5}

for c in classe.keys():
    label_train = []
    label_test = []

    class_train_path = os.path.join(dataset_train_path, c)
    class_test_path = os.path.join(dataset_test_path, c)

    img_train_path = class_train_path# +'/img'
    img_test_path = class_test_path#+'/img'

    for imag in os.listdir(img_train_path):
        label_train.append({'image_name' : imag, 'label' : classe[c]})


    for imag in os.listdir(img_test_path):
        label_test.append({'image_name' : imag, 'label' : classe[c]})


    df_train = pd.DataFrame(label_train)
    df_train.to_csv(os.path.join(class_train_path, f"{classe[c]}_label_train.csv"), index=False)
    df_test = pd.DataFrame(label_test)
    df_test.to_csv(os.path.join(class_test_path, f"{classe[c]}_label_test.csv"), index=False)



In [21]:
#the classes we are interested by
A = 'buildings'
B = 'forest'

path_img_A_train = os.path.join(dataset_train_path, A)
print(path_img_A_train)
path_img_B_train = os.path.join(dataset_train_path, B)
path_img_AB_train = os.path.join(dataset_train_path, 'buildings&forest')
path_img_A_test = os.path.join(dataset_test_path, A)
path_img_B_test = os.path.join(dataset_test_path, B)
path_img_AB_test = os.path.join(dataset_test_path, 'buildings&forest')

#create the directory if it does not exist
os.makedirs(path_img_AB_train, exist_ok=True)
os.makedirs(path_img_AB_test, exist_ok=True)

def move(init, final):
    for file in os.listdir(init):
        source_file = os.path.join(init, file)
        dest_file = os.path.join(final, file)
     #   os.rename(source_file, dest_file)
        shutil.copy2(source_file, dest_file)
  #      print(f"Fichier copié de {source_p} vers {destination_path}")



move(path_img_A_train, path_img_AB_train)
move(path_img_B_train, path_img_AB_train)

move(path_img_A_test, path_img_AB_test)
move(path_img_B_test, path_img_AB_test)

label_A_train = pd.read_csv(os.path.join(dataset_train_path, A, f"{classe[A]}_label_train.csv"))
label_B_train = pd.read_csv(os.path.join(dataset_train_path, B, f"{classe[B]}_label_train.csv"))
label_AB_train = pd.concat([label_A_train, label_B_train], ignore_index=True)
label_AB_train.to_csv(os.path.join(dataset_train_path, 'label_train.csv'), index=False)


label1=os.path.join(dataset_test_path, A, f"{classe[A]}_label_test.csv")
label2=os.path.join(dataset_test_path, B, f"{classe[B]}_label_test.csv")
label_A_test = pd.read_csv(label1)
label_B_test = pd.read_csv(label2)
label_AB_test = pd.concat([label_A_test, label_B_test], ignore_index=True)
label_AB_test.to_csv(os.path.join(dataset_test_path, 'label_test.csv'), index=False)

/content/my_intel_dataset/seg_train/seg_train/buildings


In [24]:
label_AB_train

,image_name,label
0,4332.jpg,0
1,2469.jpg,0
2,15701.jpg,0
3,2743.jpg,0
4,11197.jpg,0
...,...,...
4457,14510.jpg,1
4458,13274.jpg,1
4459,19223.jpg,1
4460,4605.jpg,1


In [36]:
import os
import torch
import pandas as pd
from torch.utils.data import Dataset
from PIL import Image

class IntelDataset(Dataset):
    def __init__(self, df, transform=None):
        """
        df: Un DataFrame filtré pour une tâche spécifique
        transform: Les transformations PyTorch (Resize, ToTensor)
        """
        # On réinitialise l'index pour éviter les trous après le filtrage par classe
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # On récupère le chemin de l'image et le label depuis vos 2 colonnes
        # (Adaptez 'image_path' et 'label' selon les vrais noms de vos colonnes)
        img_path = row['image_path']
        label = int(row['label'])

        # Chargement de l'image
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)





In [30]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# celui utilisé piur le train
print(label_AB_train)

my_transf=transforms.Compose([
    transforms.Resize((150,150)),transforms.ToTensor()]
);

dataset_AB=IntelDataset(label_AB_train,transform=my_transf)
loader_AB=DataLoader(dataset_AB, batch_size=32, shuffle=True)
print(loader_AB)

     image_name  label
0      4332.jpg      0
1      2469.jpg      0
2     15701.jpg      0
3      2743.jpg      0
4     11197.jpg      0
...         ...    ...
4457  14510.jpg      1
4458  13274.jpg      1
4459  19223.jpg      1
4460   4605.jpg      1
4461  15343.jpg      1

[4462 rows x 2 columns]


In [31]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [34]:
class EpisodicMemory:
  def __init__(self, max_memories_sample=100):
    """
    max_memory_sample : c'set le max d'image a retenir pour chaque tache passée
    """
    self.max_memories_sample=max_memories_sample
    self.memory_dataset=None   # Pas encore de TensorDataset à acceuillir

  def store_memories(self,task_dataset):
    """
    extrait un echantullon aleatoire de la tache qui veint de se terminer
    """

    # On cree un datalaoder temporaire avec comme batchsize le nombre de "souvenir" max autorisé
    temp_loader=DataLoader(task_dataset, batch_size=self.max_memories_sample,shuffle=True)


    #extraction d'un batch

# iter => actuve l'iterateur et next +> prend la "1ere boucle"
    images_from_samples,labels_sample = next(iter(temp_loader))

    if self.memory_dataset is None:
      #TensorDataste => permet de reunir les X et les y en un daatset
       #EN 0 => les x et en 1 => le y
      self.memory_dataset=TensorDataset(images_from_samples,labels_sample)

    else:
      old_labels=self.memory_dataset.tensors[1]
      old_images=self.memory_dataset.tensors[0]

      # On fusionne le passé et le present

      new_images=torch.cat([old_images,images_from_samples] )
      new_labels=torch.cat([old_labels,labels_sample])

      self.memory_dataset=TensorDataset(new_images,new_labels)

      print(f"-> Buffer A-GEM mis à jour. Taille totale de la mémoire : {len(self.memory_dataset)} images.")


    def get_reference_batch(self, batch_size):
      if self.memory_size is None:
        return None

      ref_loader=DataLoader(self.memory_dataset, batch_size=batch_size)
      return next(iter(ref_loader))




In [38]:
NB_EPOCH=3
epochs_per_task=5
memory_size_per_task=100
batch_size=32

In [40]:


def train_agrem_step(model,optimizer,criterion,current_batch,batch_ref):


  ## Initialisations

  current_imgs, current_labels=current_batch

  optimizer.zero_grad()
  current_output=model(current_imgs)
  current_loss=criterion(current_output, current_labels)
  current_loss.backward()

  #selection des parametres qii ont besoin de gradient
  param_with_grad1=[p.grad for p in model.parameters() if p.grad is not None ]


  # Transf en vecteur 1D
  g=parameters_to_vector(param_with_grad1).clone()

  #PArt 2
  img_ref,label_ref=batch_ref
  optimizer.zero_grad()

  output_ref=model(img_ref)
  loss_ref=criterion(output_ref,label_ref)
  loss_ref.backward()

  param_with_grad2=[p.grad for p in model.parameters() if p.grad is not None ]


  g_ref=parameters_to_vector(param_with_grad2).clone()

  #produit scalire entre g et g_ref

  dot_product=torch.dot(g,g_ref)

  if dot_product <0:

    # to calculate g - ( (g^t * g_ref) / (g_ref^T*g_ref) ) * g_ref
    g_proj=g - ( dot_product/torch.dot(g_ref,g_ref) ) * g_ref

    g=g_proj


  optimizer.zero_grad()

  index=0

  for p in model.parameters():

    # On regarde juste les couches qui ont besoin de gradient
    if p.requires_grad:

      # on compte le nombre d'elemnts de la couche ([3,3,4]=>3*3*4)
      num_params=p.numel()
      # on decoupe les tranches dans notre g
      tranche=g[index:index +num_params]

      p.grad=tranche.view(p.shape).clone()
      index+=num_params

  optimizer.step()

  return current_loss.item()









In [39]:
#### CODE OPTIMISATION GLOBAL  ####

import torch
import torch.nn as nn
from torch.nn.utils import parameters_to_vector,vector_to_parameters

def train_agrem_global(model, optimizer,criterion,current_batch,ref_batch):
  """
  Effectue l'optiisation d'un pas en verifiant que la contrainte est respectée
  """
  buffer_agem=EpisodicMemory(max_memories_sample=100)


## initiatlisations
 # current_imag,curent_batch=current_batch
  optimizer.zero_grad()
  model.train()

  list_of_task=[0,1]
  #
 # current_output = model(current_imag)
  for id_task in list_of_task:
    print(f"debut de la tache {id_task} ")
    current_df=label_AB_train[ label_AB_train['label']==id_task  ]

    #Dataset et loader utilisé pour la tache en question specifiquement
    current_dataset=IntelDataset(current_df,transform=my_transf)
    current_loader=DataLoader(current_dataset,batch_size=batch_size)


    for epoch in range(1,epochs_per_task):
      total_epoch_loss=0;
      for current_imgs, current_labels in current_loader:

        #recuperer le batch de reference
        batch_ref=buffer_agem.get_reference_batch(batch_size=batch_size)

        if batch_ref is not None:
          # on active l'entaiement avce projetcion car acquis à proteger
          loss= train_agrem_step(model, optimizer,criterion,current_batch,batch_ref)
        else:
          # Entrainement simple car aucun acquis à proteger
          loss = train_standard_step(model, optimizer, criterion, batch_actuel)

        #On actualise a perte totale
        total_epoch_loss+=loss

      mean_loss=total_epoch_loss/len(current_loader)
      print(f"Tâche {id_task} - Époque {epoch}/{epochs_per_task} - Perte moyenne : {mean_loss:.4f}")


    print(f"=============== FIN DE LA TACHE {id_task} : Debut remplissage du buffer ==== ")
    # on actualise le buffer avec les esilttast de la tache actuelle
    buffer_agem.store_memories(current_dataset)
  print(">>> Entraînement Continuel A-GEM terminé avec succès ! <<<")






In [ ]:
### A encore faire => La fonction de train normale####